In [1]:
from datetime import datetime
from iww_route_assignment import assign_traffic
from iww_route_assignment.io import load_euris_inputs, read_assignment_trips, write_assignment_outputs
from iww_route_assignment.simulation import LockSimulationSettings, LockSimulator
from pathlib import Path
import pandas as pd

from iww_route_assignment.simulation import inspect_trip
from iww_route_assignment import (AssignmentSettings, CostInputs, ScenarioCostParameters,
                                  VesselCatalogue, RouteCostMetric)
from iww_route_assignment.io import (IVS_SK_CODE_CROSSWALK, load_cost_rates_csv,
                                     load_fuel_provider, load_vessel_references_csv)

from time import perf_counter

In [2]:
src_dir = Path('~').expanduser() / 'projects/iww-route-assignment-sito-is/fis-crawler/output/'
data_dir = Path('~').expanduser() / 'data/ivs/iww-route-assignment'

graph_path = src_dir / 'euris-enriched/graph.pickle'
structures_dir = src_dir / 'dropins-euris-detailed'

inputs = Path("~/projects/iww-route-assignment-sito-is/iww-route-assignment/output/assign_ivs_trips/placeholder_inputs").expanduser()
vessel_path = inputs / "vessel_references.csv"
rates_path  = inputs / "cost_rates.csv"

In [3]:
fuel = pd.read_csv(inputs / "fuel_observations.csv")
fuel = fuel[fuel["vessel_profile"] == "general"].drop(columns="vessel_profile")
fuel_path = Path("/tmp/fuel_general.csv")
fuel.to_csv(fuel_path, index=False)

settings = AssignmentSettings(metric=RouteCostMetric.TRAVEL_TIME)
scenario = ScenarioCostParameters(fuel_price_eur_per_l=1.20, interest_rate_percent=3.0, cargo_waiting_time_h=1.0)

catalogue = VesselCatalogue(load_vessel_references_csv(vessel_path), class_aliases=IVS_SK_CODE_CROSSWALK)
cost_inputs = CostInputs(load_cost_rates_csv(rates_path),
                         load_fuel_provider(fuel_path, vessel_path), scenario)

In [4]:
frame = pd.read_csv(data_dir / 'ivs_trips_extended_2026.csv', index_col=0)

t = perf_counter()
euris = load_euris_inputs(graph_path, structures_dir)      # structures_dir = dropins-euris-detailed
print(f"Load EURIS: {perf_counter() - t:.2f}s")

t = perf_counter()
trips = read_assignment_trips(frame, source_dataset="ivs", format="extended_2026",
                              cargo_weight_unit="t", duplicate_trip_ids="exclude")       # departure_time comes from Eerste_Tijdstip
print(f"Load trips: {perf_counter() - t:.2f}s")

t = perf_counter()
simulator = LockSimulator.from_directory(
    structures_dir, euris.network,
    LockSimulationSettings(
        window_start=datetime(2026, 1, 1), window_end=datetime(2027, 1, 1),  # start with one week
        reroute_threshold_h=0.5, lookahead_h=None,
    ),
)
print(f"Initialize simulator: {perf_counter() - t:.2f}s")

t = perf_counter()
result = assign_traffic(trips, euris.network, cost_inputs, catalogue,
                        euris.restrictions, settings=settings, simulator=simulator)
print(f"Assign traffic: {perf_counter() - t:.2f}s")

print(result.summary["simulation"])   # counts, waits, reroutes, lock_data and placement reports
result.tables["lock_summary"].sort_values("total_wait_h", ascending=False).head(20)
write_assignment_outputs(result, "output/with_locks")

Load EURIS: 1.90s
Load trips: 103.80s


INFO:iww_route_assignment.io.locks:Read 2877 lock complexes with 3147 chambers


Initialize simulator: 1.79s


INFO:iww_route_assignment.assignment.runner:Routing 231870 of 232963 trips
INFO:iww_route_assignment.simulation.simulator:Simulated 139447 trips: 345884 lock passages, 26741 reroutes
INFO:iww_route_assignment.assignment.runner:Assignment finished: {'cost_fail': 90, 'success': 231870, 'vessel_fail': 1003}


Assign traffic: 647.25s
{'trips_simulated': 139447, 'trips_without_lock': 92423, 'trips_without_departure_time': 0, 'trips_outside_window': 0, 'lock_passages': 345884, 'oversize_passages': 6740, 'reroutes': 26741, 'mean_wait_min': 18.776148298873093, 'max_wait_min': 981.7188152552893, 'simulation_start': '2026-01-01 00:00:00', 'simulation_end': '2026-09-06 16:00:30.244290330', 'settings': {'reroute': True, 'reroute_threshold_h': 0.5, 'lookahead_h': None, 'exhaustive_search': False, 'window_start': '2026-01-01 00:00:00', 'window_end': '2027-01-01 00:00:00', 'start_side': 'A'}, 'lock_data': {'locks_in_file': 3004, 'complexes': 2877, 'chambers': 3147, 'default_operation_time_min': 30.0, 'chambers_skipped_no_dimensions': 143, 'dimensions_from_polygon': 1, 'operation_time_defaults': 62, 'linked_locks_without_chambers': 0}, 'placement': {'complexes': 2877, 'placed': 1373, 'edges_with_locks': 735, 'edges_with_several_locks': 150, 'locks_without_edge': 1504}}


INFO:pyogrio._io:Created 1,563 records
INFO:pyogrio._io:Created 2,123 records


{'trips': PosixPath('output/with_locks/voyages.parquet'),
 'trip_edges': PosixPath('output/with_locks/voyage_edges.parquet'),
 'intensity': PosixPath('output/with_locks/intensity.geoparquet'),
 'gpkg': PosixPath('output/with_locks/assignment.gpkg'),
 'rejections': PosixPath('output/with_locks/trip_rejections.csv'),
 'summary': PosixPath('output/with_locks/summary.json'),
 'planned_edges': PosixPath('output/with_locks/planned_edges.parquet'),
 'lock_passages': PosixPath('output/with_locks/lock_passages.parquet'),
 'lock_summary': PosixPath('output/with_locks/lock_summary.parquet'),
 'reroutes': PosixPath('output/with_locks/reroutes.parquet')}

In [8]:
result.tables["reroutes"].head()          # trips that changed route; pick a trip_id

trip_id = result.tables["reroutes"].loc[10000, "trip_id"]
trip = inspect_trip(result, euris.network, trip_id)

trip.groupby("route")["arc_id"].apply(list)    # planned vs sailed, edge by edge
trip.explore(column="route", cmap=["orange", "purple"])                   # map with both routes

# result.tables["reroutes"].query("trip_id == @trip_id")      # junction, expected wait, saving
# result.tables["lock_passages"].query("trip_id == @trip_id") # queue, entry and exit times per lock